# Prompting Techniques — Hands-On

**Session 7 — Applied Natural Language Processing**

This notebook explores the core prompting techniques covered in today's lecture. You will experiment with zero-shot, one-shot, and few-shot prompting, chain-of-thought reasoning, role prompting, and temperature effects — all using **sentiment analysis** as the running example.

We use **GitHub Models** for API access (free with a GitHub account). Full API coverage is Session 8.

**Prerequisites:** Set your GitHub token as an environment variable before running:
```
export GITHUB_TOKEN=your_token_here
```

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

from openai import OpenAI

# GitHub Models client — free with a GitHub account
# Full API coverage in Session 8; this is a lightweight preview
client = OpenAI(
    base_url="https://models.inference.ai.azure.com",
    api_key=os.environ.get("GITHUB_TOKEN", "set-your-github-token"),
)
MODEL = "gpt-4o-mini"

def chat(prompt, system=None, temperature=0.3, max_tokens=300):
    """Simple helper to send a prompt and get a response."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content.strip()

print("Ready.")

---

## Part 1 — Zero-Shot, One-Shot, and Few-Shot Prompting

Prompting techniques differ in how many examples (demonstrations) you provide before asking the model to perform the task.

| Technique | Examples provided | When to use |
|---|---|---|
| Zero-shot | None | General tasks, quick tests |
| One-shot | 1 per class | When you have one clear example |
| Few-shot | 3+ examples | Consistent format, domain-specific style |

We'll test all three on the same sentiment analysis input.

In [ ]:
review = "Despite the long wait, the food was absolutely delicious and the staff were incredibly friendly."

# Zero-shot: direct instruction, no examples
zero_shot_prompt = f"""Analyze the sentiment of the following text. 
Classify it as Positive, Negative, or Neutral.

Text: "{review}"
Sentiment:"""

result = chat(zero_shot_prompt)
print("Zero-shot result:")
print(result)

In [ ]:
# One-shot: one example before the target
one_shot_prompt = f"""Analyze sentiment. Classify as Positive, Negative, or Neutral.

Example:
Text: "The movie was a complete waste of time. The plot was confusing."
Sentiment: Negative

Now classify:
Text: "{review}"
Sentiment:"""

result = chat(one_shot_prompt)
print("One-shot result:")
print(result)

In [ ]:
# Few-shot: multiple examples establish the pattern clearly
few_shot_prompt = f"""Classify the sentiment of each text as Positive, Negative, or Neutral.

Text: "The new smartphone exceeded all my expectations. Camera quality is outstanding."
Sentiment: Positive

Text: "I'm so disappointed with the service. My order arrived damaged."
Sentiment: Negative

Text: "The package arrived on time. Nothing special, but it works."
Sentiment: Neutral

Text: "{review}"
Sentiment:"""

result = chat(few_shot_prompt)
print("Few-shot result:")
print(result)

**What to notice:**
- All three likely give the correct answer here — this review is unambiguous.
- Try a more ambiguous review (e.g. "The price was high but the quality justified it") and compare how each technique handles it.
- Few-shot tends to be most consistent in format — the model learns from the examples exactly how to format the output.

---

## Part 2 — Chain-of-Thought Prompting

Standard prompting asks for the answer directly. **Chain-of-thought (CoT)** prompting asks the model to reason step-by-step before answering — this significantly improves performance on complex reasoning tasks.

Two variants:
- **Standard CoT**: Provide a worked example with reasoning steps
- **Zero-shot CoT**: Add "Let's think step by step" (Kojima et al., 2022) — no examples needed

In [ ]:
# A tricky review with mixed sentiment
tricky_review = "Despite the long wait and slightly overcooked pasta, the atmosphere was wonderful and the dessert was the best I've ever had."

# Standard CoT: explicit reasoning steps in the instruction
cot_prompt = f"""Analyze the sentiment of the following text by following these steps:

1. Identify key emotional words or phrases
2. Determine if each is positive, negative, or neutral
3. Consider any context that modifies their meaning
4. Conclude with the overall sentiment

Text: "{tricky_review}"

Analysis:"""

result = chat(cot_prompt, temperature=0.1, max_tokens=400)
print("Chain-of-thought result:")
print(result)

In [ ]:
# Zero-shot CoT: just add "Let's think step by step"
zero_cot_prompt = f"""Analyze the sentiment of this text. Let's think step by step.

Text: "{tricky_review}"
"""

result = chat(zero_cot_prompt, temperature=0.1, max_tokens=400)
print("Zero-shot CoT result:")
print(result)

**What to notice:**
- CoT produces a more nuanced response — it surfaces the tension between positive and negative elements.
- The simple few-shot prompt from Part 1 might collapse mixed sentiment into one label. CoT surfaces the nuance.
- Zero-shot CoT ("Let's think step by step") is surprisingly effective for its simplicity.

---

## Part 3 — Role Prompting

Assigning the model a role or persona can improve the quality and consistency of its responses. The model draws on its training to inhabit the role.

In [ ]:
review = "The product arrived late, but the quality exceeded my expectations."

# Without role
no_role = chat(f"Analyze the sentiment: '{review}'")

# With expert role
with_role = chat(
    f"Analyze the sentiment: '{review}'",
    system="You are an expert in sentiment analysis with 10 years of experience. "
           "Provide a brief, structured analysis covering: overall sentiment, key emotional signals, "
           "and any nuance or ambiguity in the text."
)

print("Without role:")
print(no_role)
print()
print("With sentiment expert role:")
print(with_role)

**What to notice:**
- Role prompting tends to produce more structured, domain-aware responses.
- It's particularly useful when you want a consistent format across many outputs.
- The role sets implicit expectations — an "expert" produces more detailed analysis than a generic response.

---

## Part 4 — Temperature Effects

**Temperature** controls the randomness of the model's output.

| Range | Behaviour | Good for |
|---|---|---|
| 0.0 – 0.3 | Deterministic, consistent | Classification, fact extraction, evaluation |
| 0.4 – 0.7 | Balanced | General tasks |
| 0.8 – 1.0 | Creative, diverse | Brainstorming, creative writing |

Note: temperature is an **inference parameter**, not a model hyperparameter — it doesn't change the model, it changes how the model samples from its output distribution.

In [ ]:
prompt = "Write a one-sentence product tagline for a reusable coffee cup."

for temp in [0.0, 0.5, 1.0]:
    results = []
    for _ in range(3):  # run 3 times to show consistency/variation
        result = chat(prompt, temperature=temp, max_tokens=60)
        results.append(result)
    
    print(f"Temperature = {temp}:")
    for i, r in enumerate(results, 1):
        print(f"  Run {i}: {r}")
    print()

**What to notice:**
- At `temperature=0.0`: all 3 runs should produce the same (or very similar) output.
- At `temperature=1.0`: each run produces a noticeably different tagline.
- For sentiment classification, use low temperature — you want consistent labels.
- For brainstorming or creative writing, use higher temperature to get diverse outputs.

---

## Part 5 — Grammar Correction Bot (Worked Example)

A well-structured prompt uses multiple elements together: a **role**, a **task**, detailed **instructions**, and **constraints**. This example from the lecture shows a Grammar Correction Bot prompt.

This is the kind of multi-element prompt you'd build for a production use case.

In [ ]:
grammar_bot_system = """Role: You are an expert English language teacher with extensive experience in grammar correction and explanation.

Task: Correct the grammar in the given text and provide explanations for the corrections.

Instructions: Follow these steps when correcting grammar:
1. Read the entire text to understand the context
2. Identify and correct all grammatical errors
3. Explain each correction clearly and concisely
4. Preserve the original meaning

Additional guidelines:
- If a sentence is grammatically correct, state "No correction needed" and briefly explain why.
- Pay attention to: subject-verb agreement, verb tenses, pronoun usage, word order, articles and prepositions.
- Format your response as:
  Original: [original sentence]
  Corrected: [corrected sentence]
  Explanation: [what was changed and why]"""

test_sentences = [
    "She don't know what are she doing yesterday at the park.",
    "The team have been working very hard on their project since last week.",
    "Yesterday I go to the store and buyed some groceries.",
]

for sentence in test_sentences:
    print(f"Input: {sentence}")
    print()
    result = chat(sentence, system=grammar_bot_system, temperature=0.1, max_tokens=300)
    print(result)
    print("-" * 60)
    print()

**What to notice:**
- The system prompt does a lot of heavy lifting — the user message is just the raw text.
- Structured instructions (numbered steps + explicit output format) produce consistent, structured output.
- This pattern — detailed system prompt, minimal user input — is the foundation of production LLM applications.

---

## Part 6 — LLM Challenges: Live Demos

The lecture covered several LLM limitations. Let's observe them directly.

In [ ]:
# Hallucination: ask for something obscure/nonexistent
# LLMs can confidently produce plausible-sounding but false information

hallucination_prompt = """List 3 peer-reviewed papers published in 2019-2020 specifically about 
using BERT for sentiment analysis in the restaurant industry, 
including the journal name, volume, and DOI."""

result = chat(hallucination_prompt, temperature=0.3, max_tokens=400)
print("Model response:")
print(result)
print()
print("NOTE: Verify each of these references — LLMs frequently fabricate citations.")
print("This is known as hallucination: confident, plausible, but potentially false.")

In [ ]:
# Sycophancy: LLMs tend to agree with confident false claims

correct_claim = "The Eiffel Tower is located in Paris, France."
incorrect_claim = "The Eiffel Tower is located in London, England, right? I'm pretty sure about this."

for claim in [correct_claim, incorrect_claim]:
    result = chat(f"Is this correct? {claim}", temperature=0.1, max_tokens=150)
    print(f"Claim: {claim}")
    print(f"Response: {result}")
    print()

In [ ]:
# Non-determinism: same prompt, different outputs at higher temperature

prompt = "In one sentence, what is the most important thing to know about prompt engineering?"

print("Running the same prompt 5 times at temperature=0.8:")
print()
for i in range(5):
    result = chat(prompt, temperature=0.8, max_tokens=80)
    print(f"Run {i+1}: {result}")

---

## Summary

| Technique | When to use | Key parameter |
|---|---|---|
| Zero-shot | Quick tests, general tasks | None |
| One-shot | When you have one clear example | 1 example |
| Few-shot | Consistent format, domain tasks | 3+ examples |
| Chain-of-thought | Complex reasoning, nuanced tasks | Reasoning steps |
| Role prompting | Structured output, domain expertise | System prompt |
| Low temperature | Classification, evaluation, consistency | `temperature=0.0` |
| High temperature | Brainstorming, creative writing | `temperature=0.8+` |

**Key takeaway:** Prompting is iterative. Start simple (zero-shot), add examples (few-shot) if needed, add reasoning (CoT) for complex tasks, and use roles to shape output format and quality.

**Next:** `02_llm_evaluation.ipynb` — how do we evaluate whether these prompts actually work?